# 🏦 Text-to-GraphQL: Ingestion & RAG Pipeline Debugger

This notebook allows you to debug the data ingestion process into ChromaDB and verify the retrieval quality for the Text-to-GraphQL PoC.

## 1. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("/path/to/repo")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import settings
from src.context.llama_pipeline import run_ingestion_pipeline
from src.context.retriever import LlamaContextRetriever
from src.context.transforms import SourceHTMLCleaner
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logger = logging.getLogger(__name__)

/Users/hiep.t.nguyen/Desktop/Workspace/hiep/tcb-text-to-graphql/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Documents
Loading metadata markdown files from the `data/` directory.

In [2]:
data_dir = os.path.join(project_root, "data")
reader = SimpleDirectoryReader(
    input_dir=str(data_dir),
    recursive=False,
    required_exts=[".md"],
)
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {data_dir}")

Loaded 6 documents from /Users/hiep.t.nguyen/Desktop/Workspace/hiep/tcb-text-to-graphql/data


## 3. Ingestion Pipeline
Running the pipeline to transform documents into nodes and store them in ChromaDB.

In [3]:
# Setup ChromaDB
chromadb_path = os.path.join(project_root, "chroma_data")
db = chromadb.PersistentClient(path=str(chromadb_path))
chroma_collection = db.get_or_create_collection("llama_metadata")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Setup Embedding Model
embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",  # You can change to text-embedding-3-large if preferred
    api_key=settings.openai_api_key,
)

# Run Ingestion
# We use run_ingestion_pipeline because it handles metadata enrichment and deterministic chunk IDs
nodes = run_ingestion_pipeline(
    documents=documents,
    embedding_model=embed_model,
    vector_store=vector_store,
    show_progress=True
)

print(f"Generated and stored {len(nodes)} nodes.")

INFO:src.context.llama_pipeline:Built ingestion pipeline: chunk_size=1024, chunk_overlap=200


Whole document reformat failed: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}


INFO:src.context.llama_pipeline:Successfully added 126/126 nodes to vector store.
Generated and stored 126 nodes.


## 4. Debug Nodes
Examine the transformed nodes to ensure transformations (like HTML cleaning) were applied correctly.

In [4]:
from IPython.display import display, Markdown
def printmd(string):
    display(Markdown(string))

for i, node in enumerate(nodes):  # Print first 5 nodes
    print(f"--- Node {i+1} ---")
    print(f"Node ID: {node.node_id}")
    printmd(f"Source: {node.metadata.get('file_name', 'Unknown')}")
    printmd(f"Content Sample: {node.get_content()}...")
    printmd(f"Metadata: {node.metadata}")
    print("-" * 30)

--- Node 1 ---
Node ID: chunk-0-2026-03-15T08-34-53.662446Z


Source: customer-overview-dashboard-schema.md

Content Sample: ---
title: "Customer Overview Dashboard Schema"
description: "Contract adapter cho Customer Overview Dashboard, chuẩn hóa từ field thật trong Data Lake sample sang payload FE hiện tại."
created: 2026-03-05
updated: 2026-03-12
tags: [ai-frontline-poc, dashboard, schema, customer-360, urd]
status: draft
---...

Metadata: {'file_path': '/Users/hiep.t.nguyen/Desktop/Workspace/hiep/tcb-text-to-graphql/data/customer-overview-dashboard-schema.md', 'file_name': 'customer-overview-dashboard-schema.md', 'file_type': 'text/markdown', 'file_size': 9317, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14', 'header_path': '/', 'chunkId': 0, 'ingestedAt': '2026-03-15T08:34:53.662446Z'}

------------------------------
--- Node 2 ---
Node ID: chunk-1-2026-03-15T08-34-53.662446Z


...

------------------------------
--- Node 126 ---
Node ID: chunk-125-2026-03-15T08-34-53.662446Z


Source: ui-mapping.md

Content Sample: ### Time-series and NBO Fields
- **Average AUM (24 months):**   
  **Data Lake Mapping:** `aum_3m_amount + date_key (group by month) = latest 24 months`
- **NBO 1:**   
  **Data Lake Mapping:** `nbo_1`
- **NBO 2:**   
  **Data Lake Mapping:** `nbo_2`
- **NBO 3:**   
  **Data Lake Mapping:** `nbo_3`...

Metadata: {'file_path': '/Users/hiep.t.nguyen/Desktop/Workspace/hiep/tcb-text-to-graphql/data/ui-mapping.md', 'file_name': 'ui-mapping.md', 'file_type': 'text/markdown', 'file_size': 3829, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14', 'header_path': '/Overview Dashboard UI Mapping/Field Mapping/', 'chunkId': 125, 'ingestedAt': '2026-03-15T08:34:53.662446Z'}

------------------------------


## 5. Retrieval Verification
Testing semantic search against the ingested metadata.

In [5]:
# Load index from vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context, embed_model=embed_model
)

retriever = LlamaContextRetriever(index)

query = "Hỏi về số dư CASA của khách hàng"
results = retriever.retrieve(query)

print(f"Query: {query}")
print("\nRetrieved Context:")
print(results)

INFO:src.context.retriever:Nodes not provided. Attempting to load from vector store for BM25...
DEBUG:bm25s:Building index from IDs objects
INFO:src.context.retriever:BM25 retriever initialized with 674 nodes
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated queries:
Dưới đây là 3 biến thể truy vấn tìm kiếm khác nhau cho câu hỏi về số dư CASA của khách hàng, khai thác các góc nhìn khác nhau:
1. **Sử dụng từ đồng nghĩa và mở rộng phạm vi truy vấn (rộng hơn):**
*"Tổng số dư tài khoản thanh toán và tiết kiệm (CASA) hiện tại của khách hàng là bao nhiêu?"*
- Ở đây, CASA được giải thích rõ hơn là bao gồm cả tài khoản thanh toán và tiết kiệm, giúp mở rộng phạm vi tìm kiếm.
2. **Tập trung vào khía cạnh kỹ thuật, sử dụng tên trường dữ liệu trong hệ thống ngân hàng (hẹp hơn, chi tiết hơn):**
*"Truy vấn giá trị trường `current_balance` trong bảng `customer_casa_accounts` cho khách hàng có `customer_id` = X."*
- Biến thể này tập trung vào cách tru